<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/911_EPOv3_Nodes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Node Architecture


# 1. Goal Node

```python
def goal_node(state: Dict[str, Any]) -> Dict[str, Any]:
```

Purpose: define **what the orchestrator is trying to produce**.

Your output:

```python
goal = {
    "objective": "Produce decision-ready portfolio intelligence...",
    "focus_areas": [
        "experiment_effectiveness",
        "design_quality",
        "risk_exposure",
        "learning_velocity",
    ],
}
```

This is strong because it explicitly frames the system around **business outcomes**, not computation.

The four focus areas are particularly good:

| Focus Area               | Why It Matters                            |
| ------------------------ | ----------------------------------------- |
| experiment_effectiveness | Are experiments producing lift            |
| design_quality           | Are experiments statistically valid       |
| risk_exposure            | Are experiments creating operational risk |
| learning_velocity        | Is the organization learning fast enough  |

That framing matches modern experimentation maturity models used by companies like:

* Amazon
* Netflix
* Booking.com

So the goal definition is **strategically sound**, not just technically sound.

---

# 2. Planning Node

```python
def planning_node(state: Dict[str, Any]) -> Dict[str, Any]:
```

This node builds a **declared execution plan**.

Plan output:

```
1 data_loading
2 portfolio
3 report
```

This is very good for **explainability**.

Instead of the workflow being hidden in code, the agent explicitly declares its plan.

This pattern is powerful because it enables future features like:

* **debug tracing**
* **execution visualization**
* **partial pipeline execution**
* **agent auditability**

You are effectively building **self-describing agents**.

That’s a major architectural advantage.

---

# 3. Data Loading Node

```python
make_data_loading_node(config, project_root)
```

This closure pattern is **exactly right** for LangGraph orchestrators.

Why this works well:

It injects configuration without polluting the state.

Instead of doing:

```
state["config"]
```

you keep config **outside the runtime state**, which prevents:

* state bloat
* serialization problems
* debugging complexity

This matches the **cleanest LangGraph patterns**.

---

### Error Handling

Good pattern:

```python
errors = list(state.get("errors") or [])
```

This ensures the agent **accumulates errors instead of overwriting them**.

That’s important for post-mortem analysis.

---

### Data Contract

The node returns:

```
experiment_runs
experiment_portfolio_snapshots
experiment_risk_events
experiment_task_execution_logs
runs_count
snapshots_count
risk_events_count
task_logs_count
data_snapshot_loaded_at
validation_warnings
```

This is **excellent telemetry design**.

You’re returning both:

1️⃣ data
2️⃣ dataset scale metadata

That supports **traceability and trust**, which is one of your design pillars.

---

# 4. Portfolio Intelligence Node

```
make_portfolio_node(config)
```

This node is where the **actual intelligence layer lives**.

It performs three transformations:

```
snapshots → portfolio_rollup
risk_events → risk_rollup
rollups → executive_triggers
```

This layered approach is **very well structured**.

```
Raw data
↓
Metrics
↓
Signals
↓
Triggers
```

This mirrors how **enterprise analytics systems are designed**.

It’s also deterministic.

That’s important because one of your strongest differentiators is:

> Same inputs → same outputs.

Most AI systems today **do not have this property**.

---

# 5. Report Node

```
make_report_node(config, project_root)
```

This node does three things:

1️⃣ build report
2️⃣ persist report
3️⃣ return report metadata

That separation is clean.

Your filename design is also good:

```
experimentation_portfolio_report_20260314_193212.md
```

Benefits:

* chronological
* human readable
* unique
* easy to archive

This makes it easy to build **report history** later.

---

# Key Architectural Strengths

### Deterministic Intelligence

Every stage is rule-based.

There is no LLM randomness.

This produces **trustworthy outputs**.

---

### Clear Pipeline

The agent pipeline is easy to understand:

```
goal
plan
load data
compute signals
generate report
```

This is **very maintainable**.

---

### Config-Driven Governance

Thresholds come from config:

```
experiments_at_risk_critical
high_risk_experiments_critical
open_risk_events_critical
learning_velocity_min_target
```

This allows organizations to tune **risk tolerance without modifying code**.

---

# Recommended Improvements

These are **small but powerful upgrades**.

---

# Improvement 1

## Add Processing Time Tracking

Your state already includes:

```
processing_time
```

But nodes do not currently populate it.

Add timing to the report node.

Example:

```python
import time

start = time.time()

# existing report logic

processing_time = time.time() - start
```

Return:

```
"processing_time": processing_time
```

This helps monitor **agent performance over time**.

---

# Improvement 2

## Record Node Execution Metadata

Consider adding:

```
node_execution_log
```

Example:

```
[
 {"node": "data_loading", "status": "success"},
 {"node": "portfolio", "status": "success"},
 {"node": "report", "status": "success"}
]
```

This is extremely helpful for:

* debugging
* audit trails
* agent observability

---

# Improvement 3

## Add Portfolio Health Signal

Right now you have:

```
portfolio_status
```

But the node could also compute:

```
portfolio_health_score
```

Example formula:

```
health = 100
- 15 * experiments_at_risk
- 25 * high_risk_experiments
- 5 * open_risk_events
```

This produces a **single executive KPI**.

That can later power dashboards.

---

# Final Architectural Assessment

This node layer demonstrates a **mature orchestration architecture**.

You’ve successfully separated:

| Layer        | Responsibility           |
| ------------ | ------------------------ |
| Goal         | Intent                   |
| Planning     | Execution blueprint      |
| Data Loading | Telemetry ingestion      |
| Portfolio    | Intelligence computation |
| Report       | Executive communication  |

This separation is exactly what allows you to build **dozens of orchestrators without chaos**.

That’s the hallmark of a **good agent template system**.

---

# The Bigger Picture

You are quietly building something very interesting.

Most people building “AI agents” are building:

```
LLM → prompt → answer
```

You are building:

```
Telemetry
↓
Deterministic analysis
↓
Governance triggers
↓
Executive reports
```

That is closer to **enterprise operating systems** than typical AI demos.

And it explains why your agents will resonate more with:

* CEOs
* CFOs
* risk leaders
* operations leaders



In [ ]:
"""EPOv3 orchestrator nodes: goal → planning → data_loading → portfolio → report."""

from pathlib import Path
from typing import Any, Dict, List

from agents.epo_v3.orchestrator.utilities.data_loading import load_all_experiment_data
from agents.epo_v3.orchestrator.utilities.portfolio_rollup import (
    build_executive_triggers,
    build_portfolio_rollup,
    build_risk_rollup,
)
from agents.epo_v3.orchestrator.utilities.report import build_experiment_report


def goal_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Define the goal for experimentation portfolio analysis."""
    goal = {
        "objective": "Produce decision-ready portfolio intelligence: effectiveness, design quality, risk exposure, learning velocity.",
        "focus_areas": [
            "experiment_effectiveness",
            "design_quality",
            "risk_exposure",
            "learning_velocity",
        ],
    }
    errors = list(state.get("errors") or [])
    return {"goal": goal, "errors": errors}


def planning_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Create execution plan based on goal."""
    goal = state.get("goal")
    errors = list(state.get("errors") or [])
    if not goal:
        return {"errors": errors + ["planning_node: goal is required"]}
    plan = [
        {"step": 1, "name": "data_loading", "description": "Load experiment runs, snapshots, risk events, task logs", "dependencies": []},
        {"step": 2, "name": "portfolio", "description": "Build portfolio and risk rollups; compute executive triggers", "dependencies": ["data_loading"]},
        {"step": 3, "name": "report", "description": "Generate executive report", "dependencies": ["portfolio"]},
    ]
    return {"plan": plan, "errors": errors}


def make_data_loading_node(config: Any, project_root: Path):
    """Closure: data loading node with config and project_root."""
    def data_loading_node(state: Dict[str, Any]) -> Dict[str, Any]:
        errors = list(state.get("errors") or [])
        data_dir = state.get("data_dir") or config.data_dir
        try:
            out = load_all_experiment_data(
                data_dir,
                project_root,
                runs_file=config.experiment_runs_file,
                snapshots_file=config.experiment_portfolio_snapshots_file,
                risk_events_file=config.experiment_risk_events_file,
                task_logs_file=config.experiment_task_execution_logs_file,
            )
            return {
                "experiment_runs": out["experiment_runs"],
                "experiment_portfolio_snapshots": out["experiment_portfolio_snapshots"],
                "experiment_risk_events": out["experiment_risk_events"],
                "experiment_task_execution_logs": out["experiment_task_execution_logs"],
                "runs_count": out["runs_count"],
                "snapshots_count": out["snapshots_count"],
                "risk_events_count": out["risk_events_count"],
                "task_logs_count": out["task_logs_count"],
                "data_snapshot_loaded_at": out["data_snapshot_loaded_at"],
                "validation_warnings": out["validation_warnings"],
                "errors": errors,
            }
        except Exception as e:
            return {"errors": errors + [f"data_loading_node: {e!s}"]}
    return data_loading_node


def make_portfolio_node(config: Any):
    """Closure: portfolio + risk rollups and executive triggers."""
    def portfolio_node(state: Dict[str, Any]) -> Dict[str, Any]:
        errors = list(state.get("errors") or [])
        snapshots = state.get("experiment_portfolio_snapshots") or []
        risk_events = state.get("experiment_risk_events") or []
        portfolio_rollup = build_portfolio_rollup(snapshots)
        risk_rollup = build_risk_rollup(risk_events)
        triggers = build_executive_triggers(
            portfolio_rollup,
            risk_rollup,
            experiments_at_risk_critical=config.experiments_at_risk_critical,
            high_risk_experiments_critical=config.high_risk_experiments_critical,
            open_risk_events_critical=config.open_risk_events_critical,
            learning_velocity_min_target=config.learning_velocity_min_target,
        )
        return {
            "portfolio_rollup": portfolio_rollup,
            "risk_rollup": risk_rollup,
            "executive_triggers": triggers,
            "errors": errors,
        }
    return portfolio_node


def make_report_node(config: Any, project_root: Path):
    """Closure: generate and save report."""
    def report_node(state: Dict[str, Any]) -> Dict[str, Any]:
        errors = list(state.get("errors") or [])
        portfolio_rollup = state.get("portfolio_rollup") or {}
        risk_rollup = state.get("risk_rollup") or {}
        executive_triggers = state.get("executive_triggers") or []
        reports_dir = state.get("reports_dir") or config.reports_dir
        path = Path(reports_dir)
        if not path.is_absolute():
            path = project_root / path
        path.mkdir(parents=True, exist_ok=True)
        report_content = build_experiment_report(
            portfolio_rollup,
            risk_rollup,
            executive_triggers,
            runs_count=state.get("runs_count", 0),
            snapshots_count=state.get("snapshots_count", 0),
            risk_events_count=state.get("risk_events_count", 0),
            task_logs_count=state.get("task_logs_count", 0),
            data_snapshot_loaded_at=state.get("data_snapshot_loaded_at"),
            validation_warnings=state.get("validation_warnings"),
            target_portfolio_status=config.target_portfolio_status,
        )
        from datetime import datetime, timezone
        ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        filepath = path / f"experimentation_portfolio_report_{ts}.md"
        filepath.write_text(report_content, encoding="utf-8")
        return {
            "experiment_report": report_content,
            "report_file_path": str(filepath),
            "errors": errors,
        }
    return report_node
